# H2 : Linear Regression

> [!NOTE]
> This homework uses the pinned 2026 car fuel-efficiency release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.

### Dataset

For this homework, we'll use the 2026 Car Fuel Efficiency dataset. Download it from <a href='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'>here</a>.

You can do it with wget:

In [1]:
#!wget -P ../data https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

The goal of this homework is to create a regression model for predicting the car fuel efficiency <u>(column `'fuel_efficiency_mpg'`)</u>.

### Preparing the dataset 

Use only the following columns:

* `'engine_displacement'`,
* `'horsepower'`,
* `'vehicle_weight'`,
* `'model_year'`,
* `'fuel_efficiency_mpg'`


In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('../data/car_fuel_efficiency_2026.csv')

In [4]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [5]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

### EDA

* Look at the `fuel_efficiency_mpg` variable. Does it have a long tail? 


In [6]:
df["fuel_efficiency_mpg"].value_counts().tail(10)

fuel_efficiency_mpg
21.2    1
21.5    1
21.7    1
20.5    1
21.8    1
20.9    1
41.2    1
19.8    1
20.7    1
21.4    1
Name: count, dtype: int64

### Question 1
There's one column with missing values. What is it?


In [7]:
dataset_columns = ['engine_displacement', 'horsepower', 'vehicle_weight',
          'model_year', 'fuel_efficiency_mpg']

In [8]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [9]:
df.columns[df.isnull().any()].intersection(dataset_columns)

Index(['horsepower'], dtype='str')

### Question 2

What's the median (50% percentile) for variable `'horsepower'`?

In [10]:
df.horsepower.describe()

count    9123.000000
mean      254.446016
std        22.844613
min       171.000000
25%       239.000000
50%       254.000000
75%       270.000000
max       334.000000
Name: horsepower, dtype: float64

In [11]:
df.horsepower.median()

np.float64(254.0)

### Preparing and split the dataset

Shuffle the filtered dataset and create the split exactly as in the lecture:

In [12]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

> [!NOTE]
> For Q5, repeat the same block with each listed seed. For Q6, use seed 9.



### Question 3

* We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using `round(score, 3)`. This
  keeps the imputation difference visible in this release.
* Which option gives better RMSE?

From Q1, the column with missing values is `'horsepower'`. We train the model
with the code from the lessons (`train_linear_regression`, normal equation,
no regularization) and evaluate both imputation options on the validation set:

In [13]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [14]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

#### Option 1: fill missing values with 0

In [15]:
X_train = df_train[base].fillna(0).values
w0, w = train_linear_regression(X_train, y_train)

X_val = df_val[base].fillna(0).values
y_pred = w0 + X_val.dot(w)

rmse_zero = rmse(y_val, y_pred)
round(rmse_zero, 3)

np.float64(2.205)

#### Option 2: fill missing values with the mean (computed on the training set only)

In [16]:
hp_mean = df_train.horsepower.mean()

X_train = df_train[base].fillna(hp_mean).values
w0, w = train_linear_regression(X_train, y_train)

X_val = df_val[base].fillna(hp_mean).values
y_pred = w0 + X_val.dot(w)

rmse_mean = rmse(y_val, y_pred)
round(rmse_mean, 3)

np.float64(2.202)

In [17]:
print(f"RMSE (fill with 0):    {round(rmse_zero, 3)}")
print(f"RMSE (fill with mean): {round(rmse_mean, 3)}")

RMSE (fill with 0):    2.205
RMSE (fill with mean): 2.202


**Answer:** Filling with the **mean** gives the better RMSE: 2.202 (mean) vs 2.205 (0).

The mean was computed on the training partition only (`df_train.horsepower.mean()`)
and the same value was used to fill the validation set. Both models were trained
without regularization using `train_linear_regression` from the lesson.